# 9C · Lab 1 — Forecasting a Seasonal Series, Honestly
### Financial Analytics — Module 9 (Time Series Lab)

> ### 🛡️ Bias Check (lab law)
> **Look-ahead:** seasonal indices, trends and ARIMA fits use TRAINING data only; the test year is untouched until scoring. ✍️
> **Survivorship:** single continuous series. ✍️
> **Point-in-time:** synthetic first-print data. ✍️
> **Regime:** we assume the festive pattern of the training years persists into the test year — stated, not hidden. ✍️

The finale: forecast MoneyMart's monthly revenue 12 months out, with Module 6's full harness — honest split, the right baseline, walk-forward thinking, and **intervals that widen with horizon**. Four contenders:

1. **Seasonal naive** — this month = same month last year *(the baseline that respects seasonality)*
2. **SES** — Module 6's smoother *(spoiler: watch it fail, instructively)*
3. **Decompose → forecast → recompose** — the hand-built champion
4. **ARIMA** — the classical model, gently

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(9)

# Rebuild 9A's series (same seed = identical data)
months = pd.date_range("2020-04-01", periods=72, freq="MS")
t = np.arange(72)
SEASON = np.tile([-30,-25,-15,-5,10,20,60,110,45,-40,-55,-75], 6)
rev = pd.Series(400 + 6.5*t + SEASON + rng.normal(0,12,72), index=months, name="revenue_cr")

# THE SPLIT - a date, as always. Last 12 months held out.
train, test = rev.iloc[:-12], rev.iloc[-12:]
print(f"Train: {len(train)} months to {train.index[-1]:%b %Y} | Test: {len(test)} months")

---
## 1. The right baseline: seasonal naive

Plain naive ("next month = this month") is blind to seasonality — it will forecast December using November's Diwali spike. The fair floor for a seasonal series is **seasonal naive: this month = the same month, last year.**

In [ ]:
fc_snaive = train.iloc[-12:].values          # last year's 12 months, replayed

def score(pred, actual):
    e = np.asarray(actual) - np.asarray(pred)
    return {"MAE": np.abs(e).mean(), "RMSE": np.sqrt((e**2).mean())}

print("Seasonal naive:", {k: round(v,1) for k,v in score(fc_snaive, test).items()})
print("(Every model below must beat THIS, not plain naive - matching the baseline to the series' structure")
print(" is itself a forecasting skill.)")

## 2. SES: the wrong tool, instructively

Module 6's smoother tracks a *level*. It has no concept of months — so its 12-month-ahead forecast is one flat line:

In [ ]:
def ses_level(series, alpha=0.4):
    level = series.iloc[0]
    for x in series: level = alpha*x + (1-alpha)*level
    return level

fc_ses = np.repeat(ses_level(train), 12)
print("SES:", {k: round(v,1) for k,v in score(fc_ses, test).items()}, " <- much worse. WHY matters:")
print("SES isn't broken - it's answering 'what's the current level?' A flat answer to a seasonal question.")
print("Tool-vs-problem mismatch is the most common forecasting error in practice.")

## 3. The hand-built champion: decompose → forecast → recompose

Use 9A's anatomy as a *forecasting recipe*: (a) estimate trend on train, extend it 12 months (Module 6's regression); (b) estimate seasonal indices on train; (c) forecast = extended trend + seasonal index. Every part is glass-box.

In [ ]:
from scipy import stats as st

# (a) trend on TRAIN only
tt = np.arange(len(train))
lin = st.linregress(tt, train.values)
trend_fc = lin.intercept + lin.slope*np.arange(len(train), len(train)+12)

# (b) seasonal indices on TRAIN only (detrend first, then month-average)
detr = train.values - (lin.intercept + lin.slope*tt)
si = pd.Series(detr, index=train.index).groupby(train.index.month).mean()

# (c) recompose
fc_dfr = trend_fc + si.reindex(test.index.month).values
print("Decompose-forecast-recompose:", {k: round(v,1) for k,v in score(fc_dfr, test).items()})

## 4. ARIMA, gently

**ARIMA(p, d, q)** — the classical workhorse. Read the letters, skip the algebra:
- **AR(p)**: today leans on its last p values *(9B's autocorrelation, used)*
- **I(d)**: difference d times first *(9B's stationarity fix, built in)*
- **q (MA)**: today also corrects for the last q forecast errors

The **S**easonal variant adds the same three ideas at lag 12. One honest fit — order chosen by the series' anatomy, not by search:

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Anatomy says: trend (d=1), yearly season (D=1 at 12), light memory (1,0/0,1 terms)
model = SARIMAX(train, order=(1,1,1), seasonal_order=(0,1,1,12)).fit(disp=False)
fr = model.get_forecast(12)
fc_arima = fr.predicted_mean
ci = fr.conf_int(alpha=0.05)          # 95% interval - and watch what it does with horizon

print("SARIMA:", {k: round(v,1) for k,v in score(fc_arima, test).items()})

In [ ]:
# THE LEADERBOARD + the picture with intervals
board = pd.DataFrame({
    "seasonal naive": score(fc_snaive, test),
    "SES (flat)":     score(fc_ses, test),
    "decomp-recomp":  score(fc_dfr, test),
    "SARIMA":         score(fc_arima, test),
}).T.round(1).sort_values("RMSE")
print(board)

fig, ax = plt.subplots(figsize=(11, 4))
rev.iloc[-36:].plot(ax=ax, color="#94A3B8", lw=1.2, label="actual")
pd.Series(fc_dfr, index=test.index).plot(ax=ax, color="#0D9488", lw=1.8, label="decomp-recomp")
fc_arima.plot(ax=ax, color="#2563EB", lw=1.8, label="SARIMA")
ax.fill_between(test.index, ci.iloc[:,0], ci.iloc[:,1], color="#2563EB", alpha=0.12, label="SARIMA 95% interval")
ax.axvline(train.index[-1], color="black", ls=":", lw=1)
ax.set_title("12-month forecasts vs reality - and the interval WIDENING with horizon", loc="left", fontweight="bold")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

w1  = ci.iloc[0,1]-ci.iloc[0,0]; w12 = ci.iloc[-1,1]-ci.iloc[-1,0]
print(f"Interval width: month 1 = Rs {w1:.0f} cr -> month 12 = Rs {w12:.0f} cr ({w12/w1:.1f}x wider)")
print("Uncertainty COMPOUNDS with horizon. Any 12-month forecast with a constant-width band is lying.")

**Read the leaderboard the Module 6 way.** The glass-box decompose-recompose and SARIMA both beat seasonal naive — *that* clears the floor, so these models have earned their keep. SES trails badly, and you know exactly why. And the intervals widen ~like √horizon: the visual signature of honest forecasting (and the exact shape Module 10's simulations will reproduce from first principles).

### ✏️ Exercise 1 — walk-forward, seasonal edition
Re-run the contest walk-forward style: for each of the last 18 months, refit on all data before it, forecast 1 month, score. Does the ranking hold month by month, or did one model get lucky in the single split?

### ✏️ Exercise 2 — break the regime, watch the models
Inject a shock: multiply the last 6 test months by 0.85 (a demand slump). Which forecaster degrades most gracefully — the one carrying explicit structure (decomp-recomp) or SARIMA? What does the answer say about glass-box models in unstable worlds?

### ✏️ Exercise 3 — the deposit-balance transfer
Generate a deposit-balance series: `2000 + 25*t + strong March/Sep spikes (fiscal year-end) + noise`. Rerun the full pipeline — split, seasonal-naive floor, decomp-recomp, SARIMA. Nothing in the code should need more than the seasonal patterns changed. *That* is the point: the pipeline is the skill; the series is a parameter.

---
## Lab 1 complete
You can now take any seasonal business series - revenue, deposits, loan disbursals, transaction volumes - apart, test its memory, forecast it against the right floor, and ship intervals that widen honestly. This is the most broadly employable lab in Part D: every stream on the Module 8 map runs this loop somewhere.

*AI disclosure: ______*

In [ ]:
# workspace
